# Notebook 5 — Multi-Agent Debate Refinement (MADR) with Heterogeneous Models

**Goal:** replace a single model's fact-checking judgment with a structured,
multi-round debate among **different models from different providers**, and
compare the refined result against Notebook 1's closed-book and open-book
baselines on the same PolitiFact claims.

## Why debate — and why heterogeneous models?

A single model reviewing its own explanation tends to ratify its own errors:
same weights, same blind spots. The **Multi-Agent Debate Refinement (MADR)**
framework generates *faithful* fact-checking explanations by forcing an
explanation through adversarial review before it is trusted, in five
structured steps:

1. **Explain** — a zero-shot model generates an initial explanation for the
   claim's veracity.
2. **Debate** — two *independent* debater agents analyze the explanation for
   errors. One debates freely; the other is guided by a structured **error
   typology**: factual inconsistency, unsupported inference, missing context,
   logical fallacy. Both debaters have **web search** enabled (OpenRouter's
   `web` plugin) so their critiques can check claims against live evidence.
3. **Exchange** — the debaters read each other's critiques and refine their
   own, surfacing contradictions neither caught alone.
4. **Judge** — a judge agent reviews the exchanged feedback and establishes
   consensus on which critiques are valid.
5. **Refine** — a refiner agent integrates the consensus into a finalized,
   verified explanation with a final veracity label and confidence.

Crucially, each role is served by a **different underlying model** — the
second-best current model from each provider — routed through the
**OpenRouter SDK**. Disagreements between heterogeneous models carry real
information: they stem from different training corpora and inductive biases,
not from sampling the same distribution twice.

| Role | Model (via OpenRouter) | Web search |
|------|------------------------|------------|
| Explainer | `openai/gpt-5.6-terra` | – |
| Debater (general) | `anthropic/claude-opus-4.8` | yes |
| Debater (typology) | `google/gemini-3.5-flash` | yes |
| Judge | `x-ai/grok-4.3` | – |
| Refiner | `openai/gpt-5.6-terra` | – |

(The roster lives in `toolkit.config.DEBATE_MODEL_ROSTER` — one place to swap
models.) Verdicts use the same 7-label vocabulary as Notebook 1: the six
PolitiFact Truth-O-Meter labels plus *"Not enough information"*.


In [ ]:
import os
from pathlib import Path

# Notebooks live in notebooks/; run from the repo root so relative paths like
# ./data behave identically in VS Code and classic Jupyter. The local
# `toolkit` package is installed (editable) in the .venv, so no sys.path hacks.
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)
print(f"Working directory: {Path.cwd()}")

import json

import pandas as pd
from pydantic import BaseModel, Field

from toolkit.config import (
    DEBATE_MODEL_ROSTER,
    FACTCHECK_DATA_PATH,
    OPENROUTER_WEB_PLUGIN,
    get_openrouter_client,
)
from toolkit.metrics import MULTI_CLASS_ORDER, TO_TERNARY, VERACITY_VERDICTS

client = get_openrouter_client()

# Same filter + seeded sample as Notebook 1, so results join on the
# factcheck_analysis_link key.
RANDOM_SEED = 303
claims_df = pd.read_parquet(FACTCHECK_DATA_PATH)
claims_df = claims_df[claims_df["verdict"].isin(VERACITY_VERDICTS)]
sample_df = claims_df.sample(n=10, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Sampled {len(sample_df)} claims (seed {RANDOM_SEED})")

nb01_path = Path("data/nb01_results.csv")
baseline = None
if nb01_path.exists():
    nb01 = pd.read_csv(nb01_path)
    baseline = nb01.pivot_table(
        index="factcheck_analysis_link", columns="condition", values="predicted", aggfunc="first"
    )
    print(f"Loaded Notebook 1 baselines for {len(baseline)} claims")
else:
    print("No data/nb01_results.csv found — run Notebook 1 for the baseline comparison.")

print("\nDebate roster:")
for role, model in DEBATE_MODEL_ROSTER.items():
    print(f"  {role:<18} -> {model}")


## 1. One schema per debate stage


In [ ]:
VERACITY_SCALE = ", ".join(MULTI_CLASS_ORDER)


class InitialExplanation(BaseModel):
    veracity: str = Field(description=f"One of: {VERACITY_SCALE}.")
    explanation: str = Field(description="Initial zero-shot explanation for the veracity verdict.")


class DebaterCritique(BaseModel):
    identified_errors: list[str] = Field(description="Errors or gaps identified in the explanation.")
    error_types: list[str] = Field(default_factory=list, description="Error typology labels, if using the structured typology debater.")
    proposed_revisions: str = Field(description="Suggested revisions to address the identified errors.")


class JudgeConsensus(BaseModel):
    agreed_errors: list[str] = Field(description="Errors both debaters converged on after exchanging feedback.")
    unresolved_disagreements: list[str] = Field(default_factory=list, description="Points of contradiction the debaters could not resolve.")
    consensus_summary: str = Field(description="Judge's synthesis of the debate for the refiner to act on.")


class RefinedExplanation(BaseModel):
    final_veracity: str = Field(description=f"Final veracity label after debate; one of: {VERACITY_SCALE}.")
    final_explanation: str = Field(description="Finalized, verified explanation integrating consensus feedback.")
    confidence: float = Field(description="Confidence rating between 0.0 and 1.0.")


## 2. Structured outputs over OpenRouter

OpenRouter forwards a `response_format` JSON schema to the underlying
provider, but schema enforcement varies across providers. So we defend in
depth:

1. request a **strict JSON schema** derived from the Pydantic model
   (`model_json_schema()` + `additionalProperties: false`);
2. validate the returned text with `Model.model_validate_json`;
3. if validation fails (e.g., the model wrapped its JSON in prose or a code
   fence), extract the first balanced `{...}` object and validate that.

Roles that should verify facts against live evidence get OpenRouter's
model-agnostic **web plugin** (`plugins=[{"id": "web"}]`) attached to the
call.


In [ ]:
def strict_json_schema(schema_cls):
    schema = schema_cls.model_json_schema()
    schema["additionalProperties"] = False
    return schema


def extract_first_json(text):
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object found in model output.")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    raise ValueError("Unbalanced JSON object in model output.")


def response_text(response):
    choice = response.choices[0]
    message = choice.message if hasattr(choice, "message") else choice["message"]
    content = message.content if hasattr(message, "content") else message["content"]
    if isinstance(content, list):  # some providers return content parts
        content = "".join(part.get("text", "") if isinstance(part, dict) else str(part) for part in content)
    return content or ""


def call_agent(role, system_prompt, user_prompt, schema_cls, web_search=False):
    model = DEBATE_MODEL_ROSTER[role]
    kwargs = {}
    if web_search:
        kwargs["plugins"] = OPENROUTER_WEB_PLUGIN
    response = client.chat.send(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema_cls.__name__,
                "strict": True,
                "schema": strict_json_schema(schema_cls),
            },
        },
        temperature=0.2,
        stream=False,
        **kwargs,
    )
    raw = response_text(response)
    try:
        return schema_cls.model_validate_json(raw)
    except Exception:
        return schema_cls.model_validate_json(extract_first_json(raw))


## 3. The five MADR stages as functions

Each stage receives only the artifacts it needs (not the full transcript) to
keep token usage flat as the debate grows.


In [ ]:
ERROR_TYPOLOGY = (
    "factual inconsistency (contradicts known facts), "
    "unsupported inference (conclusion not backed by stated evidence), "
    "missing context (omits information that changes the interpretation), "
    "logical fallacy (invalid reasoning step)"
)


def claim_text(row):
    return f'Claim (made by {row.statement_originator}): "{row.statement}"'


def generate_initial_explanation(row):
    return call_agent(
        "explainer",
        f"You are a fact-checker. Rate the claim with exactly one of these labels: {VERACITY_SCALE}. "
        "Explain your verdict clearly and concisely.",
        f"{claim_text(row)}\n\nProvide your veracity verdict and explanation.",
        InitialExplanation,
    )


def run_debater(role, row, explanation, use_typology=False):
    if use_typology:
        system = (
            "You are a critical reviewer of fact-checking explanations. Analyze "
            "the explanation for errors using this typology, labeling each error "
            f"you find with its type: {ERROR_TYPOLOGY}. "
            "Use your web search access to verify factual assertions."
        )
    else:
        system = (
            "You are a critical reviewer of fact-checking explanations. Identify "
            "any errors, gaps, or weaknesses in the explanation, reasoning freely. "
            "Use your web search access to verify factual assertions."
        )
    return call_agent(
        role,
        system,
        f"{claim_text(row)}\n\nVerdict: {explanation.veracity}\n"
        f"Explanation under review:\n{explanation.explanation}\n\n"
        "List the errors you identify and propose revisions.",
        DebaterCritique,
        web_search=True,
    )


def exchange_feedback(role, row, own_critique, other_critique, use_typology=False):
    system = (
        "You are a debater refining your critique of a fact-checking "
        "explanation after reading a fellow debater's independent critique. "
        "Keep points you still stand behind, drop points the other critique "
        "convincingly undermines, adopt valid points you missed, and flag any "
        "contradictions between the two critiques."
    )
    if use_typology:
        system += f" Continue labeling errors with the typology: {ERROR_TYPOLOGY}."
    return call_agent(
        role,
        system,
        f"{claim_text(row)}\n\n"
        f"YOUR earlier critique:\n{own_critique.model_dump_json(indent=2)}\n\n"
        f"THE OTHER debater's critique:\n{other_critique.model_dump_json(indent=2)}\n\n"
        "Produce your refined critique.",
        DebaterCritique,
        web_search=True,
    )


def judge_consensus(row, explanation, critique_a, critique_b):
    return call_agent(
        "judge",
        "You are an impartial judge reviewing two refined critiques of a "
        "fact-checking explanation. Decide which criticisms are valid, which "
        "disagreements remain unresolved, and summarize the consensus for a "
        "refiner to act on.",
        f"{claim_text(row)}\n\nOriginal verdict: {explanation.veracity}\n"
        f"Original explanation:\n{explanation.explanation}\n\n"
        f"Debater A (general) refined critique:\n{critique_a.model_dump_json(indent=2)}\n\n"
        f"Debater B (typology) refined critique:\n{critique_b.model_dump_json(indent=2)}",
        JudgeConsensus,
    )


def refine_explanation(row, explanation, consensus):
    return call_agent(
        "refiner",
        f"You are a fact-checking editor. Rate claims with exactly one of these labels: {VERACITY_SCALE}. "
        "Rewrite the explanation so it addresses every agreed error from the "
        "debate consensus. Change the veracity label only if the consensus "
        "justifies it, and report your confidence (0.0-1.0).",
        f"{claim_text(row)}\n\nOriginal verdict: {explanation.veracity}\n"
        f"Original explanation:\n{explanation.explanation}\n\n"
        f"Debate consensus:\n{consensus.model_dump_json(indent=2)}",
        RefinedExplanation,
    )


## 4. The full pipeline, with every intermediate step printed

~9 API calls per claim across 4 providers, so we debate a small sample
(`N_CLAIMS` below) — raise it if you have time and budget.


In [ ]:
def show(header, obj):
    print(f"\n--- {header} " + "-" * max(0, 60 - len(header)))
    print(obj.model_dump_json(indent=2))


def run_madr_pipeline(row, verbose=True):
    roster = DEBATE_MODEL_ROSTER
    print("=" * 78)
    print(f'CLAIM: "{row.statement}"')
    print("=" * 78)

    # Step 1 — initial zero-shot explanation
    explanation = generate_initial_explanation(row)
    if verbose:
        show(f"STEP 1 · Explainer ({roster['explainer']})", explanation)

    # Step 2 — two independent debaters (one guided by the error typology)
    critique_a = run_debater("debater_general", row, explanation, use_typology=False)
    critique_b = run_debater("debater_typology", row, explanation, use_typology=True)
    if verbose:
        show(f"STEP 2 · Debater A ({roster['debater_general']})", critique_a)
        show(f"STEP 2 · Debater B ({roster['debater_typology']})", critique_b)

    # Step 3 — debaters exchange feedback and refine their critiques
    refined_a = exchange_feedback("debater_general", row, critique_a, critique_b)
    refined_b = exchange_feedback("debater_typology", row, critique_b, critique_a, use_typology=True)
    if verbose:
        show(f"STEP 3 · Debater A refined ({roster['debater_general']})", refined_a)
        show(f"STEP 3 · Debater B refined ({roster['debater_typology']})", refined_b)

    # Step 4 — judge establishes consensus
    consensus = judge_consensus(row, explanation, refined_a, refined_b)
    if verbose:
        show(f"STEP 4 · Judge ({roster['judge']})", consensus)

    # Step 5 — refiner integrates consensus into the final explanation
    final = refine_explanation(row, explanation, consensus)
    if verbose:
        show(f"STEP 5 · Refiner ({roster['refiner']})", final)

    return {"initial": explanation, "consensus": consensus, "final": final}


In [ ]:
N_CLAIMS = 2  # ~9 calls per claim; increase if time/budget allow

madr_results = []
for row in sample_df.head(N_CLAIMS).itertuples():
    try:
        result = run_madr_pipeline(row)
        madr_results.append({
            "factcheck_analysis_link": row.factcheck_analysis_link,
            "statement": row.statement,
            "gold": row.verdict,
            "initial_veracity": result["initial"].veracity,
            "final_veracity": result["final"].final_veracity,
            "final_confidence": result["final"].confidence,
            "final_explanation": result["final"].final_explanation,
            "agreed_errors": " | ".join(result["consensus"].agreed_errors),
        })
    except Exception as e:
        print(f"\n! MADR pipeline failed for claim '{row.statement[:50]}...': {e}")
        print("  (Check model availability on OpenRouter; swap the roster entry in toolkit/toolkit/config.py if needed.)")

madr_df = pd.DataFrame(madr_results)
madr_df.to_csv("data/nb05_madr_results.csv", index=False)
print(f"\nSaved {len(madr_df)} MADR results -> data/nb05_madr_results.csv")


## 5. MADR vs. the single-model baselines

Per claim: the **gold** PolitiFact verdict, Notebook 1's **closed-book** and
**open-book** single-model verdicts, and the **MADR** verdict after
heterogeneous debate. The ternary collapse (`toolkit.metrics.TO_TERNARY`)
makes agreement easier to eyeball than the full 6-point scale.


In [ ]:
comparison = madr_df[["factcheck_analysis_link", "statement", "gold", "final_veracity"]].copy()
comparison = comparison.rename(columns={"final_veracity": "madr"})

if baseline is not None:
    comparison["closed_book"] = comparison["factcheck_analysis_link"].map(baseline.get("closed_book"))
    comparison["open_book"] = comparison["factcheck_analysis_link"].map(baseline.get("open_book"))
else:
    print("Run Notebook 1 first to include the baseline columns.")

label_cols = [c for c in ["gold", "closed_book", "open_book", "madr"] if c in comparison.columns]
for col in label_cols:
    comparison[f"{col}_ternary"] = comparison[col].map(TO_TERNARY)

comparison["statement"] = comparison["statement"].str.slice(0, 70)
comparison.drop(columns=["factcheck_analysis_link"])


In [ ]:
# Read one refined explanation end-to-end
if len(madr_df):
    row = madr_df.iloc[0]
    print(f'CLAIM: "{row.statement}"')
    print(f"gold={row.gold} | initial={row.initial_veracity} | "
          f"final={row.final_veracity} (confidence {row.final_confidence:.2f})")
    print(f"\nAgreed errors from the debate:\n  {row.agreed_errors}")
    print(f"\nFinal explanation:\n{row.final_explanation}")


## Takeaways

- **Deliberation is a quality-control layer**: even when the label does not
  change, the refined explanation is typically more hedged, better scoped,
  and stripped of unsupported inferences — the debate's value shows up in the
  *explanation's faithfulness*, not just the verdict.
- **Heterogeneity is the active ingredient.** Watch Steps 2–3: the
  typology-guided debater and the free debater reliably surface *different*
  error classes, and the exchange step is where contradictions get resolved.
- **Cost scales linearly** (~9 calls/claim, two of them with live web
  search). MADR is an auditing and explanation-generation tool, not something
  you run on every claim at ingest time.

## Where to go from here

- Feed Notebook 2's *live* articles in as evidence for the debaters.
- Swap roster entries in `toolkit.config.DEBATE_MODEL_ROSTER` and measure how
  much the judge's provider changes outcomes.
- Scale `N_CLAIMS` and score MADR with `toolkit.metrics.SCENARIOS`, exactly
  as in Notebook 1.
